# Experiment 1: LoRA fine-tuning with a fixed LR and a fixed rank


Model: microsoft/Phi-4-mini-instruct and Qwen2.5 3B

Dataset: Commonsense reasoning (commonsense_170k)

Results:
- Training loss curves (Muon vs AdamW)
- Singular value spectrum of LoRA product BA throughout training


---
## Cell 1: Install Libraries

In [ ]:
!pip install -q transformers peft datasets accelerate trl
!pip install --upgrade Pillow

---
## Cell 2: Load Muon Optimizer

In [ ]:
%run /home/ubuntu/thesis-storage/muon.ipynb

---
## Cell 4: Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from torch.utils.data import DataLoader
from accelerate import Accelerator
import copy
import os

accelerator = Accelerator()

# Device
device = accelerator.device
print('Using device:', device)

---
## Cell 5: Configuration

All hyperparameters in one place. Change things here only.

In [ ]:

MODEL_NAME = 'microsoft/Phi-4-mini-instruct'  # Repalce with model


LORA_RANK    = 16       # rank of A and B matrices
LORA_ALPHA   = 32       # scaling factor (usually 2x rank)
LORA_DROPOUT = 0.05
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj']


MAX_STEPS    = 7989   # total training steps
BATCH_SIZE   = 8       # per device
MAX_SEQ_LEN  = 512      # max token length
GRAD_ACCUM   = 8       # gradient accumulation steps

# Apart from LR rest hyper params either std or borrowed from Keller or Moonlight paper
MUON_LR          = 0.002
MUON_MOMENTUM    = 0.95
MUON_WD          = 0.1
MUON_UPDATE_SCALE = 0.3

ADAMW_LR = 3e-4
ADAMW_WD = 0.1


SVD_TRACK_EVERY = 200   # track singular values


SAVE_DIR = '/home/ubuntu/thesis-storage/experiment_1(1)_results' #Replace it with your file path
os.makedirs(SAVE_DIR, exist_ok=True)

print('Config set.')


---
## Cell 6: Load Model and Tokenizer


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False,
)

# Add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=False,
)
print('Parameters:', sum(p.numel() for p in model.parameters()) / 1e9, 'B')

---
## Cell 7: Apply LoRA

Freezes the base model. Only A and B adapter matrices will be trained.
This is what Muon and AdamW will optimize.

In [ ]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    task_type=TaskType.CAUSAL_LM,
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Confirm base weights are frozen
frozen = sum(1 for p in model.parameters() if not p.requires_grad)
trainable = sum(1 for p in model.parameters() if p.requires_grad)
print(f'Frozen params: {frozen}')
print(f'Trainable params (LoRA A and B only): {trainable}')

---
## Cell 8: Load Dataset

Commonsense 170k — same dataset used in the Riemannion paper.

In [ ]:
dataset = load_dataset('zwhe99/commonsense_170k', split='train')

---
## Cell 9: Tokenize Dataset

In [ ]:
def tokenize(example):
    # Format as instruction + response
    text = f"### Instruction:\n{example['instruction']}\n### Response:\n{example['output']}"
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        return_tensors='pt',
    )
    tokens['labels'] = tokens['input_ids'].clone()
    return {k: v.squeeze(0) for k, v in tokens.items()}

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
tokenized.set_format('torch')

dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)

---
## Cell 10: SVD Tracker

At every SVD_TRACK_EVERY steps, it computes the singular values of
the LoRA product B @ A for each layer.

Under AdamW: top-heavy distribution (a few big values).
Under Muon: uniform distribution (all values similar).

In [ ]:
def track_svd(model, step):
    """
    For each LoRA layer, compute singular values of B @ A.
    """
    sv_dict = {}

    for name, module in model.named_modules():
        # Find LoRA layers - lora_A and lora_B
        if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
            # Get A and B matrices
            A = module.lora_A['default'].weight  # shape: (rank, in_features)
            B = module.lora_B['default'].weight  # shape: (out_features, rank)

            # Compute product BA
            with torch.no_grad():
                BA = (B @ A).float()
                sv = torch.linalg.svdvals(BA).cpu().numpy()

            sv_dict[name] = sv

    return sv_dict


def compute_sv_uniformity(sv_array):
    """
    Measures how uniform the singular values are.
    Returns a score between 0 and 1.
    1 = perfectly uniform (Muon ideal)
    0 = completely top-heavy (AdamW typical)
    """
    sv = sv_array / (sv_array.sum() + 1e-8)
    entropy = -np.sum(sv * np.log(sv + 1e-8))
    max_entropy = np.log(len(sv))
    return entropy / max_entropy

---
## Cell 11: Training Function


In [ ]:
def train(
    model,
    dataloader,
    optimizer_type='adamw',
    max_steps=MAX_STEPS,
    grad_accum=GRAD_ACCUM,
):
    print(f'\n===== Training with {optimizer_type.upper()} =====')

    # Set up optimizer
    if optimizer_type == 'muon':
        muon_params, adamw_params = get_muon_and_adamw_params(model)
        optimizer = [Muon(muon_params, lr=MUON_LR, momentum=MUON_MOMENTUM,
                         weight_decay=MUON_WD, update_scale=MUON_UPDATE_SCALE)]
        if len(adamw_params) > 0:
            optimizer.append(torch.optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=ADAMW_WD))
    else:
        optimizer = [torch.optim.AdamW(model.parameters(), lr=ADAMW_LR, weight_decay=ADAMW_WD)]

    # Prepare optimizers only - model and dataloader already prepared
    optimizer = [accelerator.prepare(opt) for opt in optimizer]

    loss_history = []
    sv_history   = {}

    model.train()
    step = 0
    data_iter = iter(dataloader)

    while step < max_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        input_ids      = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels         = batch['labels']

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss / grad_accum
        accelerator.backward(loss)

        if (step + 1) % grad_accum == 0:
            for opt in optimizer:
                opt.step()
                opt.zero_grad()

        loss_val = loss.item() * grad_accum
        loss_history.append((step, loss_val))

        if step % SVD_TRACK_EVERY == 0:
            sv_dict = track_svd(accelerator.unwrap_model(model), step)
            sv_history[step] = sv_dict
            if accelerator.is_main_process:
                print(f'Step {step:4d} | Loss: {loss_val:.4f} | SVD tracked.')

        step += 1

    if accelerator.is_main_process:
        print(f'Final loss: {loss_val:.4f}')

    return loss_history, sv_history

print('Training function defined.')

---
## Cell 12: Run AdamW 


In [ ]:
# Prepare model and dataloader once — shared across both runs
model, dataloader = accelerator.prepare(model, dataloader)

# Save initial weights so we can reset for Muon run
initial_state = copy.deepcopy(accelerator.unwrap_model(model).state_dict())

# Train with AdamW
adamw_losses, adamw_svd = train(
    model,
    dataloader,
    optimizer_type='adamw',
    max_steps=MAX_STEPS,
)

torch.save({'losses': adamw_losses, 'svd': adamw_svd},
           os.path.join(SAVE_DIR, 'adamw_results.pt'))


---
## Cell 13: Reset Model and Run Muon

In [ ]:
# Reset to initial weights
accelerator.unwrap_model(model).load_state_dict(initial_state)

# Re-enable gradients for LoRA params after reset
for name, param in accelerator.unwrap_model(model).named_parameters():
    if 'lora_' in name:
        param.requires_grad_(True)

# Train with Muon
muon_losses, muon_svd = train(
    model,
    dataloader,
    optimizer_type='muon',
    max_steps=MAX_STEPS,
)

torch.save({'losses': muon_losses, 'svd': muon_svd},
           os.path.join(SAVE_DIR, 'muon_results.pt'))


---
## Cell 14: Loss Curves


In [ ]:
adamw_steps = [x[0] for x in adamw_losses]
adamw_vals  = [x[1] for x in adamw_losses]
muon_steps  = [x[0] for x in muon_losses]
muon_vals   = [x[1] for x in muon_losses]

plt.figure(figsize=(10, 5))
plt.plot(adamw_steps, adamw_vals, label='AdamW', color='blue', alpha=0.8)
plt.plot(muon_steps,  muon_vals,  label='Muon',  color='orange', alpha=0.8)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Training Loss: Muon vs AdamW (LoRA, Phi-4-mini, Rank 16)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'loss_curves.png'), dpi=150)
plt.show()

---
## Cell 15: Singular Values Plot

In [ ]:
def plot_sv_evolution(sv_history, title, save_path):
    """
    Plots singular value evolution over training steps.
    Picks the first LoRA layer as representative.
    Each line = one singular value tracked over time.
    """
    steps = sorted(sv_history.keys())
    if not steps:
        print('No SVD data.')
        return

    # Get first layer name
    first_layer = list(sv_history[steps[0]].keys())[0]

    # Collect singular values over time for that layer
    sv_over_time = []
    for s in steps:
        if first_layer in sv_history[s]:
            sv_over_time.append(sv_history[s][first_layer])

    if not sv_over_time:
        print('No data for first layer.')
        return

    sv_over_time = np.array(sv_over_time)  # shape: (num_steps, rank)

    plt.figure(figsize=(10, 5))
    for i in range(sv_over_time.shape[1]):
        plt.plot(steps[:len(sv_over_time)], sv_over_time[:, i], alpha=0.7)
    plt.xlabel('Training Step')
    plt.ylabel('Singular Value')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()


# Plot AdamW SVD evolution
plot_sv_evolution(
    adamw_svd,
    title='Singular Value Evolution — AdamW (expect top-heavy)',
    save_path=os.path.join(SAVE_DIR, 'svd_adamw.png')
)

# Plot Muon SVD evolution
plot_sv_evolution(
    muon_svd,
    title='Singular Value Evolution — Muon (expect uniform)',
    save_path=os.path.join(SAVE_DIR, 'svd_muon.png')
)